# Kalp Hastalığı Verisi — Regresyon Analizi**Veri:** heart.csv (UCI Heart Disease)**Hedefler:** `thalach` (regresyon), `target` (lojistik)

## 1. Veriyi Yükleme ve Keşif

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.model_selection import train_test_splitfrom sklearn.linear_model import LinearRegression, LogisticRegressionfrom sklearn.preprocessing import StandardScalerfrom sklearn.metrics import (mean_squared_error, r2_score, mean_absolute_error,                             accuracy_score, confusion_matrix, classification_report,                             roc_auc_score, roc_curve)plt.rcParams['figure.figsize'] = (10, 6)plt.rcParams['font.size'] = 12sns.set_style('whitegrid')import warningswarnings.filterwarnings('ignore')

In [ ]:
# CSV dosyasını okudf = pd.read_csv('heart.csv')df.head()

In [ ]:
# Sütun isimlerini ve veri tiplerini inceledf.info()

In [ ]:
# İstatistiksel özetdf.describe().T

### Değişken Sözlüğü| Sütun | Anlamı | Tür ||---|---|---|| `age` | Yaş (yıl) | Sürekli || `sex` | Cinsiyet (1 = erkek, 0 = kadın) | Kategorik || `cp` | Göğüs ağrısı tipi (0–3) | Kategorik || `trestbps` | Dinlenme kan basıncı (mm Hg) | Sürekli || `chol` | Serum kolesterol (mg/dl) | Sürekli || `fbs` | Açlık kan şekeri > 120 mg/dl (1/0) | Kategorik || `restecg` | Dinlenme EKG sonucu (0–2) | Kategorik || `thalach` | **Ulaşılan maksimum kalp atış hızı** | Sürekli → **HEDEF** || `exang` | Egzersize bağlı anjina (1/0) | Kategorik || `oldpeak` | Egzersize bağlı ST depresyonu | Sürekli || `slope` | ST segmentinin eğimi (0–2) | Kategorik || `ca` | Floroskopide boyanan ana damar sayısı (0–4) | Sayısal || `thal` | Talasemi durumu (0–3) | Kategorik || `target` | Kalp hastalığı (1 = var, 0 = yok) | İkili |

## 2. Veri Temizliği

In [ ]:
# Eksik değer kontrolüprint('Veri boyutu:', df.shape)print('\nEksik değerler:')print(df.isnull().sum())

In [ ]:
# Tekrar eden (duplike) satır kontrolü# NOT: Bu veri seti orijinal 303 satırlık UCI verisinin çoğaltılmış (upsampled) halidir.print('Tekrar eden satır sayısı:', df.duplicated().sum())df = df.drop_duplicates().reset_index(drop=True)print('Temizlik sonrası veri boyutu:', df.shape)

In [ ]:
# Mantıksız (fizyolojik olarak imkânsız) değer kontrolüfor col in ['trestbps', 'chol', 'thalach']:    print(f'{col}: min={df[col].min()}, max={df[col].max()}, sıfır sayısı={(df[col] == 0).sum()}')# Sürekli ve kategorik değişkenleri ayırsurekli = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']kategorik = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal', 'target']print('\nSürekli değişkenler:', surekli)print('Kategorik değişkenler:', kategorik)

## 3. Keşifçi Veri Analizi (EDA)

In [ ]:
# Korelasyon matrisiplt.figure(figsize=(12, 10))numeric_df = df.select_dtypes(include=[np.number])corr = numeric_df.corr()sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True)plt.title('Korelasyon Matrisi')plt.tight_layout()plt.show()

In [ ]:
# Hedef değişken thalach ile en yüksek korelasyonlu değişkenlertarget = 'thalach'corr_with_target = corr[target].sort_values(ascending=False)print('thalach ile Korelasyonlar:')print(corr_with_target)print('\nMutlak değere göre en güçlü 5 ilişki:')print(corr_with_target.drop(target).abs().sort_values(ascending=False).head(5))

In [ ]:
# thalach dağılımıplt.figure(figsize=(12, 5))plt.subplot(1, 2, 1)sns.histplot(df[target], bins=25, kde=True)plt.title(f'{target} Dağılımı')plt.xlabel('Maksimum Kalp Atış Hızı')plt.subplot(1, 2, 2)sns.boxplot(y=df[target])plt.title(f'{target} Kutu Grafiği')plt.tight_layout()plt.show()print(f'Ortalama: {df[target].mean():.2f} | Medyan: {df[target].median():.2f} | Std: {df[target].std():.2f}')

In [ ]:
# Yaş ile maksimum kalp atış hızı ilişkisi (hastalık durumuna göre renklendirilmiş)plt.figure(figsize=(10, 6))sns.scatterplot(data=df, x='age', y='thalach', hue='target', palette={0: 'steelblue', 1: 'crimson'}, alpha=0.7, s=60)plt.xlabel('Yaş')plt.ylabel('Maksimum Kalp Atış Hızı (thalach)')plt.title('Yaş - Maksimum Kalp Atış Hızı İlişkisi')plt.legend(title='Kalp Hastalığı', labels=['Yok (0)', 'Var (1)'])plt.grid(True, alpha=0.3)plt.show()

## 4. Basit Doğrusal Regresyon

### 4.1 Yaş → Maksimum Kalp Atış HızıFizyolojideki klasik kural `Maksimum Kalp Hızı ≈ 220 − Yaş`. Korelasyon matrisinde de `age` ile `thalach` arasındaki ilişki negatif.

In [ ]:
# Basit regresyon için değişkenlerX_simple = df[['age']].valuesy = df['thalach'].values# Eğitim ve test setlerine ayır (%80 eğitim, %20 test)X_train, X_test, y_train, y_test = train_test_split(    X_simple, y, test_size=0.2, random_state=42)print(f'Eğitim seti boyutu: {X_train.shape[0]}')print(f'Test seti boyutu: {X_test.shape[0]}')

In [ ]:
# Modeli oluştur ve eğitmodel_simple = LinearRegression()model_simple.fit(X_train, y_train)# Tahmin yapy_pred_train = model_simple.predict(X_train)y_pred_test = model_simple.predict(X_test)# Model parametreleribeta_0 = model_simple.intercept_beta_1 = model_simple.coef_[0]print(f'Kesim Noktası (β₀): {beta_0:.4f}')print(f'Eğim (β₁): {beta_1:.4f}')print('\nModel Denklemi:')print(f'thalach = {beta_0:.4f} + ({beta_1:.4f}) × age')print(f'\nYorum: Yaş 1 yıl arttığında maksimum kalp atış hızı ortalama {abs(beta_1):.2f} bpm düşmektedir.')

In [ ]:
# Tahmin vs gerçek değerler (test setinden ilk 10 gözlem)results = pd.DataFrame({    'Yaş': X_test.flatten(),    'Gerçek thalach': y_test,    'Tahmin Edilen thalach': y_pred_test.round(2),    'Fark (Artık)': (y_test - y_pred_test).round(2)})print('Test Seti Tahminleri (ilk 10):')print(results.head(10).to_string(index=False))

In [ ]:
# Model performans metriklerirmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))mae_train = mean_absolute_error(y_train, y_pred_train)mae_test = mean_absolute_error(y_test, y_pred_test)r2_train = r2_score(y_train, y_pred_train)r2_test = r2_score(y_test, y_pred_test)metrics = pd.DataFrame({    'Metrik': ['RMSE', 'MAE', 'R²'],    'Eğitim': [f'{rmse_train:.3f}', f'{mae_train:.3f}', f'{r2_train:.3f}'],    'Test': [f'{rmse_test:.3f}', f'{mae_test:.3f}', f'{r2_test:.3f}']})print('Model Performansı:')print(metrics.to_string(index=False))

In [ ]:
# Görselleştirme: Regresyon doğrusuplt.figure(figsize=(10, 6))# Tüm veri noktalarıplt.scatter(X_simple, y, color='steelblue', alpha=0.5, label='Gerçek Veri', s=50)# Regresyon doğrusu (x sıralı)x_line = np.linspace(X_simple.min(), X_simple.max(), 100).reshape(-1, 1)y_line = model_simple.predict(x_line)plt.plot(x_line, y_line, color='red', linewidth=2,         label=f'Regresyon Doğrusu: y = {beta_0:.2f} + ({beta_1:.2f})x')# Karşılaştırma: klasik "220 - yaş" formülüplt.plot(x_line, 220 - x_line, color='green', linestyle='--', linewidth=2, label='Klasik Formül: 220 - Yaş')plt.xlabel('Yaş')plt.ylabel('Maksimum Kalp Atış Hızı (thalach)')plt.title('Basit Doğrusal Regresyon: Yaş → Maksimum Kalp Atış Hızı')plt.legend()plt.grid(True, alpha=0.3)plt.show()

In [ ]:
# Artık (Residual) analiziresiduals = y_test - y_pred_testplt.figure(figsize=(12, 4))plt.subplot(1, 2, 1)plt.scatter(y_pred_test, residuals, alpha=0.7, s=50)plt.axhline(y=0, color='red', linestyle='--')plt.xlabel('Tahmin Edilen Değer')plt.ylabel('Artık (Residual)')plt.title('Artık Grafiği (Heteroskedastisite Kontrolü)')plt.grid(True, alpha=0.3)plt.subplot(1, 2, 2)sns.histplot(residuals, bins=15, kde=True)plt.xlabel('Artık')plt.title('Artık Dağılımı (Normallik Kontrolü)')plt.grid(True, alpha=0.3)plt.tight_layout()plt.show()print(f'Artıkların ortalaması: {np.mean(residuals):.4f}')print(f'Artıkların standart sapması: {np.std(residuals):.4f}')

## 5. Çoklu Doğrusal Regresyon

### 5.1 Özellik Seçimi ve Çoklu Bağlantı Kontrolü

In [ ]:
# thalach ile ilişkisi en güçlü değişkenleri seçelimprint('thalach ile mutlak korelasyonlar (büyükten küçüğe):')print(corr_with_target.drop('thalach').abs().sort_values(ascending=False))# Seçilen özellikler (yüksek korelasyonlu + klinik olarak anlamlı)# NOT: 'target' (hastalık etiketi) özellik olarak KULLANILMIYOR — o ayrı bir modelin hedefi.selected_features = ['age', 'oldpeak', 'exang', 'ca', 'slope', 'cp', 'trestbps', 'chol']available_features = [f for f in selected_features if f in df.columns]print(f'\nKullanılacak özellikler ({len(available_features)} adet): {available_features}')

In [ ]:
# Çoklu bağlantı (multicollinearity) kontrolü - VIF hesabıX_all = df[available_features].valuesprint('VIF Değerleri:')for i, feat in enumerate(available_features):    X_others = np.delete(X_all, i, axis=1)    y_current = X_all[:, i]    r2_i = LinearRegression().fit(X_others, y_current).score(X_others, y_current)    vif = 1 / (1 - r2_i)    durum = 'OK' if vif < 5 else ('DİKKAT' if vif < 10 else 'SORUNLU')    print(f'  {feat:<10}: {vif:>6.2f}  [{durum}]')

### 5.2 Model Kurulumu ve Eğitim

In [ ]:
# Çoklu regresyon için veri hazırlığıX_multi = df[available_features].valuesy_multi = df['thalach'].values# Eğitim ve test setlerine ayırX_train_m, X_test_m, y_train_m, y_test_m = train_test_split(    X_multi, y_multi, test_size=0.2, random_state=42)print(f'Eğitim seti: {X_train_m.shape}')print(f'Test seti: {X_test_m.shape}')

In [ ]:
# Çoklu doğrusal regresyon modelini oluştur ve eğitmodel_multi = LinearRegression()model_multi.fit(X_train_m, y_train_m)# Tahminy_pred_train_m = model_multi.predict(X_train_m)y_pred_test_m = model_multi.predict(X_test_m)# Model katsayılarıprint('Kesim Noktası (Intercept):', round(model_multi.intercept_, 4))print('\nKatsayılar:')for feat, coef in zip(available_features, model_multi.coef_):    print(f'  {feat:<10}: {coef:>8.4f}')print('\nModel Denklemi:')eq = f'thalach = {model_multi.intercept_:.2f}'for feat, coef in zip(available_features, model_multi.coef_):    eq += f' + ({coef:.2f} × {feat})'print(eq)

In [ ]:
# Model performansırmse_train_m = np.sqrt(mean_squared_error(y_train_m, y_pred_train_m))rmse_test_m = np.sqrt(mean_squared_error(y_test_m, y_pred_test_m))mae_train_m = mean_absolute_error(y_train_m, y_pred_train_m)mae_test_m = mean_absolute_error(y_test_m, y_pred_test_m)r2_train_m = r2_score(y_train_m, y_pred_train_m)r2_test_m = r2_score(y_test_m, y_pred_test_m)metrics_multi = pd.DataFrame({    'Metrik': ['RMSE', 'MAE', 'R²'],    'Eğitim': [f'{rmse_train_m:.3f}', f'{mae_train_m:.3f}', f'{r2_train_m:.3f}'],    'Test': [f'{rmse_test_m:.3f}', f'{mae_test_m:.3f}', f'{r2_test_m:.3f}']})print('Çoklu Regresyon Model Performansı:')print(metrics_multi.to_string(index=False))

In [ ]:
# İki modeli karşılaştır (Test seti üzerinden)comparison = pd.DataFrame({    'Model': ['Basit Regresyon (age)', 'Çoklu Regresyon'],    'RMSE': [f'{rmse_test:.3f}', f'{rmse_test_m:.3f}'],    'MAE': [f'{mae_test:.3f}', f'{mae_test_m:.3f}'],    'R²': [f'{r2_test:.3f}', f'{r2_test_m:.3f}']})print('Model Karşılaştırması (Test Seti):')print(comparison.to_string(index=False))

In [ ]:
# Gerçek vs Tahmin karşılaştırması (Çoklu Regresyon)plt.figure(figsize=(10, 6))plt.scatter(y_test_m, y_pred_test_m, alpha=0.7, s=60)plt.plot([y_test_m.min(), y_test_m.max()], [y_test_m.min(), y_test_m.max()],         'r--', linewidth=2, label='Mükemmel Tahmin (y=x)')plt.xlabel('Gerçek thalach')plt.ylabel('Tahmin Edilen thalach')plt.title('Çoklu Regresyon: Gerçek vs Tahmin')plt.legend()plt.grid(True, alpha=0.3)plt.show()

In [ ]:
# Standartlaştırılmış katsayılar ile özellik önemi# NOT: Ham katsayılar farklı birimlerde olduğu için doğrudan karşılaştırılamaz#      (örn. chol mg/dl, exang 0/1). Bu yüzden önce standartlaştırıyoruz.scaler = StandardScaler()X_multi_scaled = scaler.fit_transform(X_multi)model_std = LinearRegression().fit(X_multi_scaled, y_multi)feature_importance = pd.DataFrame({    'Özellik': available_features,    'Ham Katsayı': model_multi.coef_,    'Std Katsayı': model_std.coef_,    '|Std Katsayı|': np.abs(model_std.coef_)}).sort_values('|Std Katsayı|', ascending=True)print(feature_importance.to_string(index=False))plt.figure(figsize=(10, 6))colors = ['green' if c > 0 else 'red' for c in feature_importance['Std Katsayı']]plt.barh(feature_importance['Özellik'], feature_importance['Std Katsayı'], color=colors, alpha=0.7)plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)plt.xlabel('Standartlaştırılmış Katsayı')plt.title('Özelliklerin Maksimum Kalp Atış Hızı Üzerindeki Etkisi')for i, v in enumerate(feature_importance['Std Katsayı']):    plt.text(v + (0.15 if v > 0 else -0.15), i, f'{v:.2f}', va='center',             ha='left' if v > 0 else 'right')plt.grid(True, alpha=0.3)plt.tight_layout()plt.show()

In [ ]:
# Çoklu regresyon artık analiziresiduals_m = y_test_m - y_pred_test_mplt.figure(figsize=(12, 4))plt.subplot(1, 2, 1)plt.scatter(y_pred_test_m, residuals_m, alpha=0.7, s=50)plt.axhline(y=0, color='red', linestyle='--')plt.xlabel('Tahmin Edilen Değer')plt.ylabel('Artık')plt.title('Artık Grafiği (Çoklu Regresyon)')plt.grid(True, alpha=0.3)plt.subplot(1, 2, 2)sns.histplot(residuals_m, bins=15, kde=True)plt.xlabel('Artık')plt.title('Artık Dağılımı')plt.grid(True, alpha=0.3)plt.tight_layout()plt.show()print(f'Artıkların ortalaması: {np.mean(residuals_m):.4f}')print(f'Artıkların standart sapması: {np.std(residuals_m):.4f}')

## 6. Lojistik Regresyon (target)Veri setinin asıl hedef değişkeni `target` ikili olduğu için doğrusal regresyon yerine lojistik regresyon kullanılıyor.

In [ ]:
# Lojistik regresyon için veri hazırlığıX_clf = df.drop(columns=['target'])y_clf = df['target'].valuesclf_features = X_clf.columns.tolist()X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(    X_clf.values, y_clf, test_size=0.2, random_state=42, stratify=y_clf)# Ölçeklendirme (lojistik regresyon için önemli)scaler_clf = StandardScaler().fit(X_train_c)X_train_c_s = scaler_clf.transform(X_train_c)X_test_c_s = scaler_clf.transform(X_test_c)model_log = LogisticRegression(max_iter=1000, random_state=42)model_log.fit(X_train_c_s, y_train_c)y_pred_c = model_log.predict(X_test_c_s)y_prob_c = model_log.predict_proba(X_test_c_s)[:, 1]print(f'Doğruluk (Accuracy): {accuracy_score(y_test_c, y_pred_c):.3f}')print(f'ROC-AUC: {roc_auc_score(y_test_c, y_prob_c):.3f}')print('\nSınıflandırma Raporu:')print(classification_report(y_test_c, y_pred_c, target_names=['Hasta Değil (0)', 'Hasta (1)']))

In [ ]:
# Karmaşıklık matrisi ve ROC eğrisifig, axes = plt.subplots(1, 2, figsize=(14, 5))cm = confusion_matrix(y_test_c, y_pred_c)sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0],            xticklabels=['Tahmin: 0', 'Tahmin: 1'],            yticklabels=['Gerçek: 0', 'Gerçek: 1'])axes[0].set_title('Karmaşıklık Matrisi (Confusion Matrix)')fpr, tpr, _ = roc_curve(y_test_c, y_prob_c)axes[1].plot(fpr, tpr, linewidth=2, label=f'ROC (AUC = {roc_auc_score(y_test_c, y_prob_c):.3f})')axes[1].plot([0, 1], [0, 1], 'r--', linewidth=1, label='Rastgele Tahmin')axes[1].set_xlabel('Yanlış Pozitif Oranı (FPR)')axes[1].set_ylabel('Doğru Pozitif Oranı (TPR)')axes[1].set_title('ROC Eğrisi')axes[1].legend()axes[1].grid(True, alpha=0.3)plt.tight_layout()plt.show()

In [ ]:
# Lojistik regresyon katsayıları (hastalık riskine etkisi)log_importance = pd.DataFrame({    'Özellik': clf_features,    'Katsayı': model_log.coef_[0],    'Odds Oranı': np.exp(model_log.coef_[0])}).sort_values('Katsayı')print(log_importance.to_string(index=False))plt.figure(figsize=(10, 6))colors = ['green' if c > 0 else 'red' for c in log_importance['Katsayı']]plt.barh(log_importance['Özellik'], log_importance['Katsayı'], color=colors, alpha=0.7)plt.axvline(x=0, color='black', linewidth=0.5)plt.xlabel('Katsayı (standartlaştırılmış)')plt.title('Özelliklerin Kalp Hastalığı Riskine Etkisi')plt.grid(True, alpha=0.3)plt.tight_layout()plt.show()

## 7. Bulgular**Veri temizliği:** 1025 satırın 723'ü tekrar eden satır. Bu, orijinal UCI verisinin (303 satır) çoğaltılmış versiyonu. Duplikeler temizlenmezse aynı hasta hem eğitim hem test setine düşüyor ve test skorları yapay olarak şişiyor. Analiz 302 benzersiz gözlem üzerinde yapıldı.**Basit regresyon:** `thalach = 204.09 − 1.01 × age`. Bulunan eğim, fizyolojideki "220 − yaş" kuralına yakın. Test R² 0.136.**Çoklu regresyon:** Test R² 0.220. En güçlü katkı `age`, `exang` ve `slope` değişkenlerinden. Tüm VIF değerleri 5'in altında.**Lojistik regresyon:** Test doğruluk 0.803, ROC-AUC 0.871.**Not:** Tıbbi verilerde bireysel farklılık yüksek olduğu için R² değerlerinin düşük kalması beklenen bir durum.

In [ ]:
# Tüm modelleri özetleprint('=' * 60)print('KALP HASTALIĞI VERİSİ - REGRESYON ANALİZİ ÖZETİ')print('=' * 60)print(f'Veri boyutu (duplikeler temizlendikten sonra): {df.shape[0]} gözlem, {df.shape[1]} değişken')print('\n1. BASİT DOĞRUSAL REGRESYON')print(f'   Hedef: thalach (maksimum kalp atış hızı)')print(f'   Özellik: age')print(f'   Denklem: thalach = {beta_0:.2f} + ({beta_1:.2f}) × age')print(f'   Test R²  : {r2_test:.3f}')print(f'   Test RMSE: {rmse_test:.3f}')print(f'   Test MAE : {mae_test:.3f}')print('\n2. ÇOKLU DOĞRUSAL REGRESYON')print(f'   Hedef: thalach')print(f'   Özellikler: {available_features}')print(f'   Test R²  : {r2_test_m:.3f}')print(f'   Test RMSE: {rmse_test_m:.3f}')print(f'   Test MAE : {mae_test_m:.3f}')print('\n3. MODEL KARŞILAŞTIRMASI')better_model = 'Çoklu Regresyon' if r2_test_m > r2_test else 'Basit Regresyon'print(f'   R² değerine göre daha iyi model: {better_model}')print(f'   R² artışı: {(r2_test_m - r2_test):.3f}')print('\n4. LOJİSTİK REGRESYON (BONUS)')print(f'   Hedef: target (kalp hastalığı var/yok)')print(f'   Test Doğruluk: {accuracy_score(y_test_c, y_pred_c):.3f}')print(f'   Test ROC-AUC : {roc_auc_score(y_test_c, y_prob_c):.3f}')print('=' * 60)